<a href="https://colab.research.google.com/github/Nikolai-N484/Data201_NikolaiN/blob/main/Week7/Week_7_Part2_Feature_Selection_Linear_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3: Feature Selection for Linear Regression (Python)
This notebook supports a 75-minute class on **which predictors to include** in a linear regression model.

## Bridging for R users
R tools you already know:
- Fit: `lm(y ~ x1 + x2, data=df)`
- Compare: `AIC(m1, m2)`, `BIC(m1, m2)`
- Nested models / partial F-test: `anova(m_small, m_big)`
- VIF: `car::vif(m)`

Python equivalents used here:
- Fit: `statsmodels.OLS(y, X).fit()`
- Compare: `model.aic`, `model.bic`, `model.rsquared_adj`
- Partial F-test: `m_big.compare_f_test(m_small)`
- VIF: compute from regressions (function below)

**Key distinction:** Module 2 was *model form*. Today is *predictor inclusion*.


## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations

np.random.seed(12)


## 1. Teaching dataset (true predictors + redundant + noise + ID)
We simulate data so that:
- `x1`, `x2`, and `z` are truly predictive
- `x1_dup` is redundant with `x1` (multicollinearity)
- `noise1`, `noise2` are irrelevant
- `id` is an ID-like variable that should be excluded

In [ ]:
n = 220
x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 1, n)
z = np.random.choice([0, 1], size=n)
x1_dup = x1 + np.random.normal(0, 0.10, n)
noise1 = np.random.normal(0, 1, n)
noise2 = np.random.normal(0, 1, n)
id_col = np.arange(1, n+1)
y = 4 + 2.5*x1 - 1.8*x2 + 1.2*z + np.random.normal(0, 1.2, n)

df = pd.DataFrame({
    'y': y,
    'x1': x1,
    'x2': x2,
    'z': z,
    'x1_dup': x1_dup,
    'noise1': noise1,
    'noise2': noise2,
    'id': id_col
})
df.head()

### R equivalent
```r
# df <- data.frame(y, x1, x2, z, x1_dup, noise1, noise2, id)
```

## 2. Remove obvious junk (IDs / leakage) and define candidates

In [ ]:
candidates = ['x1','x2','z','x1_dup','noise1','noise2']
candidates

## 3. Redundancy check: correlation matrix

In [ ]:
df[candidates].corr()

### R equivalent
```r
# cor(df[, candidates])
```

## 4. Multicollinearity check: VIF
VIF is based on how well each predictor can be predicted from the others.
Higher VIF ⇒ more multicollinearity.

In [ ]:
def vif_table(dataframe, features):
    out = []
    for f in features:
        X = dataframe[[c for c in features if c != f]]
        X = sm.add_constant(X)
        y = dataframe[f]
        r2 = sm.OLS(y, X).fit().rsquared
        vif = 1.0 / (1.0 - r2) if r2 < 0.999999 else np.inf
        out.append((f, r2, vif))
    return pd.DataFrame(out, columns=['feature','R2_from_others','VIF']).sort_values('VIF', ascending=False)

vif_table(df, candidates)

### R equivalent
```r
# library(car)
# m <- lm(y ~ x1 + x2 + z + x1_dup + noise1 + noise2, data=df)
# vif(m)
```

**Decision point:** Drop one of the redundant predictors (`x1` vs `x1_dup`).
We keep `x1` and drop `x1_dup` for interpretability and stability.

In [ ]:
candidates_pruned = ['x1','x2','z','noise1','noise2']
vif_table(df, candidates_pruned)

## 5. Baseline model and key criteria: adjusted R², AIC, BIC

In [ ]:
X_base = sm.add_constant(df[candidates_pruned])
m_base = sm.OLS(df['y'], X_base).fit()
pd.Series({
    'R2': m_base.rsquared,
    'Adj_R2': m_base.rsquared_adj,
    'AIC': m_base.aic,
    'BIC': m_base.bic
}).round(4)

### R equivalent
```r
# m_base <- lm(y ~ x1 + x2 + z + noise1 + noise2, data=df)
# summary(m_base)$adj.r.squared
# AIC(m_base); BIC(m_base)
```

## 6. Subset comparison (teaching demo)
For teaching, we do best-subset search over 5 predictors and compare by BIC/AIC/Adj R².
In real projects, you would compare a **small set of motivated candidates** rather than brute force.

In [ ]:
def fit_subset(features):
    X = sm.add_constant(df[list(features)])
    m = sm.OLS(df['y'], X).fit()
    return {
        'features': tuple(features),
        'k': len(features),
        'Adj_R2': m.rsquared_adj,
        'AIC': m.aic,
        'BIC': m.bic,
        'model': m
    }

results = []
for k in range(1, len(candidates_pruned)+1):
    for feats in combinations(candidates_pruned, k):
        results.append(fit_subset(feats))

res_df = pd.DataFrame([{k:v for k,v in r.items() if k!='model'} for r in results])
res_df.sort_values('BIC').head(10)

Best model by each criterion:

In [ ]:
best = pd.concat([
    res_df.sort_values('BIC').head(1).assign(criterion='Best BIC'),
    res_df.sort_values('AIC').head(1).assign(criterion='Best AIC'),
    res_df.sort_values('Adj_R2', ascending=False).head(1).assign(criterion='Best Adj R2')
])[['criterion','features','k','Adj_R2','AIC','BIC']]
best

### R equivalents
```r
# m_full <- lm(y ~ x1 + x2 + z + noise1 + noise2, data=df)
# step(m_full, direction='both')                  # AIC stepwise
# step(m_full, direction='both', k=log(nrow(df))) # BIC-like stepwise
```

## 7. Partial F-test logic (nested models)
Compare reduced vs full:
- Reduced: `y ~ x1 + x2 + z`
- Full: `y ~ x1 + x2 + z + noise1 + noise2`
A large p-value suggests the added terms do not earn their complexity.

In [ ]:
m_red = sm.OLS(df['y'], sm.add_constant(df[['x1','x2','z']])).fit()
m_full = m_base
F_stat, p_val, df_diff = m_full.compare_f_test(m_red)
pd.Series({'F': F_stat, 'p_value': p_val, 'df_diff': df_diff})

### R equivalent
```r
# m_red <- lm(y ~ x1 + x2 + z, data=df)
# m_full <- lm(y ~ x1 + x2 + z + noise1 + noise2, data=df)
# anova(m_red, m_full)
```

## 8. Final model (example)
Combine evidence (AIC/BIC/Adj R² + multicollinearity + domain reasoning) to choose a final predictor set.

In [ ]:
m_final = m_red
pd.Series({'Adj_R2': m_final.rsquared_adj, 'AIC': m_final.aic, 'BIC': m_final.bic}).round(4)

## Exit ticket (concept check)
1) Why is R² alone a poor criterion for feature selection?
2) When might BIC select a smaller model than AIC?
3) What does a high VIF indicate and what is a reasonable response?
4) Give one domain-driven reason to keep a predictor even if it is not statistically strong.
